# LOAD DATA GENDER IN METADATA TO TABLE OF REFERENCE

In [2]:
import pandas as pd

# === Step 1: Load files ===
gender_file = "/home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/QUALITY_ASSURANCE/REF_100/TRIM_50/compare_gender.csv"
metadata_file = "/home/dan_pham/Public/NIPTorrent/STATISTIC/DATA/METADATA1.csv"

gender = pd.read_csv(gender_file)
metadata = pd.read_csv(metadata_file)

# === Step 2: Clean column names ===
gender.columns = gender.columns.str.strip()
metadata.columns = metadata.columns.str.strip()

# === Step 3: Extract SAMPLE ID + BARCODE from GENDER.csv ===
# sample format = 24PC1151_021
gender["SAMPLE ID"] = gender["sample"].str.split("_").str[0]
gender["BARCODE_NUM"] = gender["sample"].str.split("_").str[1]
gender["IONXPRESS BARCODE"] = "IonXpress_" + gender["BARCODE_NUM"]

# === Step 4: Select metadata columns ===
meta_subset = metadata[["SAMPLE ID", "IONXPRESS BARCODE", "GENDER"]]

# === Step 5: Merge ===
merged = pd.merge(
    gender,
    meta_subset,
    on=["SAMPLE ID", "IONXPRESS BARCODE"],
    how="left",
    suffixes=("", "_meta"),
)

# === Step 6: Fill missing gender ===
merged["GENDER"] = merged["GENDER"].combine_first(merged["GENDER_meta"])

# If Gender column (F/M) is empty, fill based on metadata's XX/XY
merged.loc[merged["Gender"].isna() & merged["GENDER"].notna(), "Gender"] = \
    merged["GENDER"].map({"XX": "F", "XY": "M"})

# === Step 7: Drop helper columns ===
merged = merged.drop(columns=["BARCODE_NUM", "GENDER_meta"])

# === Step 8: Save output ===
merged.to_csv(gender_file, index=False)

print("✅ Completed! GENDER.csv updated with metadata gender values.")


✅ Completed! GENDER.csv updated with metadata gender values.
